# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is described via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant Dataset
dataset = mlc.Dataset(croissant_url)

# Access and print Dataset metadata summary
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")

## 2. Data Overview
List available record sets, fields, and their @id values.
We use the Croissant metadata schema to locate the available record sets. If record sets or fields are nested, they are always referenced by their unique `@id`.

In [ ]:
# Display available record sets and their fields (@id)
record_sets = list(dataset.metadata.record_sets)

print("Available Record Sets:")
for rs in record_sets:
    print(f"- @id: {rs.id}, name: {rs.name}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - @id: {field.id}, name: {field.name}, dataType: {field.data_type}")
    print()

## 3. Data Extraction
Let's load data for selected record sets into Pandas DataFrames. We use each record set's `@id` and show how to access columns/fields. Update the list of record set `@id`s as appropriate for your analysis.

In [ ]:
# List all record set IDs: pick all or a subset as needed
record_set_ids = [r.id for r in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f'Loaded DataFrame for record_set @id: {rs_id}')
        print(f'Columns: {dataframes[rs_id].columns.tolist()}')
        print(dataframes[rs_id].head(), '\n')
    else:
        print(f'No records found for record_set @id: {rs_id}')

## 4. Exploratory Data Analysis (EDA)
We'll choose a record set and a numeric field (referenced by its `@id`) for basic filtering and normalization. Replace `<record_set_id>` and `<numeric_field_id>` with actual IDs from above. Grouping and aggregation by a categorical field is also demonstrated if applicable.

In [ ]:
# Example IDs (replace with actual IDs found above for your case):
example_record_set_id = record_set_ids[0] if record_set_ids else None
df = dataframes[example_record_set_id].copy() if example_record_set_id else pd.DataFrame()

# Display all column (field) names and @ids
print(f"Columns available in DataFrame for {example_record_set_id}:")
print(df.columns.tolist())

# Let's pick a likely numeric field (adjust @id as appropriate). We'll use the first numeric column found.
numeric_field_id = None
for rs in record_sets:
    if rs.id == example_record_set_id:
        for field in rs.fields:
            if field.data_type in ['Integer', 'Float', 'Number']:
                if field.id in df.columns:
                    numeric_field_id = field.id
                    print(f"Chose numeric field: {field.name} (@id={field.id}) with type {field.data_type}")
                    break
        break

# If a numeric field was found, perform filtering and normalization
if numeric_field_id:
    # Remove rows with missing values in this field
    filtered_df = df[df[numeric_field_id].notnull()].copy()
    threshold = filtered_df[numeric_field_id].mean()  # Example: filter above average
    filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]
    print(f"Filtered records with @{numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())
    # Normalization (Z-score)
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    # Try grouping by a categorical field if available
    group_field_id = None
    for rs in record_sets:
        if rs.id == example_record_set_id:
            for field in rs.fields:
                if field.data_type in ['Text', 'String'] and field.id in df.columns:
                    group_field_id = field.id
                    print(f"Using {field.name} (@id={field.id}) as group field.")
                    break
            break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"\nGrouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print('No numeric field found for demonstration of EDA.')

## 5. Visualization
We can now visualize distributions and relationships in the data for selected fields using matplotlib or seaborn. Here we show a histogram and a boxplot for the chosen numeric field, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and not filtered_df.empty:
    plt.figure(figsize=(14, 5))
    plt.subplot(1, 2, 1)
    sns.histplot(filtered_df[numeric_field_id], kde=True)
    plt.title(f"Histogram of {numeric_field_id}")
    plt.xlabel(numeric_field_id)

    plt.subplot(1, 2, 2)
    sns.boxplot(x=filtered_df[numeric_field_id])
    plt.title(f"Boxplot of {numeric_field_id}")

    plt.tight_layout()
    plt.show()
else:
    print("No numeric data available for visualization.")

## 6. Conclusion
We loaded the FAIR² mass spectrometry protein dataset with `mlcroissant`, explored the schema via record set and field @ids, and demonstrated simple filtering, normalization, and visualization of numerical data. For deep analysis, review all record sets and fields as referenced by their Croissant `@id` for reliable programmatic access.

> **Tip:** For real analyses, select fields with biological significance (e.g., protein abundances or MW), use `@id` mapping as above, and consult the Croissant schema for documentation of each field's meaning.